In [1]:
import os
import pandas as pd
from google.colab import files

# 1. Criar a pasta 'src' e o ficheiro 'data_processing.py' se não existirem nesta sessão (M)
if not os.path.exists("src/data_processing.py"):
    os.makedirs("src", exist_ok=True)
    code_src = """import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

def load_and_clean_data(df_input, target_col='Threat Level'):
    df = df_input.dropna().copy()
    id_cols = [c for c in df.columns if 'id' in c.lower() or 'date' in c.lower() or 'timestamp' in c.lower()]
    if id_cols:
        df = df.drop(columns=id_cols)
    return df

def prepare_pipeline(df, target_col='Threat Level'):
    if target_col not in df.columns:
        target_col = df.columns[-1]

    X = df.drop(columns=[target_col])
    y = df[target_col]

    label_encoders = {}
    for col in X.select_dtypes(include=['object', 'category']).columns:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le

    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y.astype(str))

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return {
        'X_train_scaled': X_train_scaled,
        'X_test_scaled': X_test_scaled,
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler,
        'target_encoder': target_encoder,
        'label_encoders': label_encoders,
        'feature_names': list(X.columns)
    }
"""
    with open("src/data_processing.py", "w") as f:
        f.write(code_src)
    print("Módulo 'src/data_processing.py' gerado automaticamente para a sessão atual!")

# 2. Carregar e preparar os dados
if 'X_train_scaled' not in globals():
    from src.data_processing import load_and_clean_data, prepare_pipeline

    if 'df' not in globals():
        print("Por favor, faz o upload do ficheiro CSV do dataset:")
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        df = pd.read_csv(file_name)

    df_clean = load_and_clean_data(df)
    prep_data = prepare_pipeline(df_clean)

    X_train_scaled = prep_data['X_train_scaled']
    X_test_scaled = prep_data['X_test_scaled']
    y_train = prep_data['y_train']
    y_test = prep_data['y_test']

# 3. Modelos de Classificação Supervisionada (M)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

classifiers = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Naïve Bayes (Gaussian)": GaussianNB(),
    "Support Vector Machine (SVM)": SVC(kernel='rbf', random_state=42, probability=True),
    "Random Forest (Bagging)": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost (Boosting)": AdaBoostClassifier(random_state=42),
    "Gradient Boosting (Boosting)": GradientBoostingClassifier(random_state=42)
}

if has_xgb:
    classifiers["XGBoost (Boosting)"] = XGBClassifier(random_state=42, eval_metric='logloss')

results_base = []

print("\n=== PARTE 3: TREINO DOS MODELOS SUPERVISIONADOS (M) ===\n")
for name, model in classifiers.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results_base.append({
        "Modelo": name,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão (Weighted)": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall (Weighted)": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "F1-Score (Weighted)": f1_score(y_test, y_pred, average='weighted', zero_division=0)
    })

df_results_base = pd.DataFrame(results_base).sort_values(by="F1-Score (Weighted)", ascending=False)
display(df_results_base)

Módulo 'src/data_processing.py' gerado automaticamente para a sessão atual!
Por favor, faz o upload do ficheiro CSV do dataset:


Saving archive.zip to archive.zip

=== PARTE 3: TREINO DOS MODELOS SUPERVISIONADOS (M) ===



,Modelo,Acurácia,Precisão (Weighted),Recall (Weighted),F1-Score (Weighted)
7,Gradient Boosting (Boosting),0.225000,0.228751,0.225000,0.224612
5,Random Forest (Bagging),0.196667,0.196848,0.196667,0.196363
3,Naïve Bayes (Gaussian),0.203333,0.217556,0.203333,0.190919
4,Support Vector Machine (SVM),0.191667,0.190834,0.191667,0.187725
1,K-Nearest Neighbors (KNN),0.188333,0.189494,0.188333,0.186199
8,XGBoost (Boosting),0.183333,0.183131,0.183333,0.183055
2,Árvore de Decisão,0.183333,0.182268,0.183333,0.182494
0,Regressão Logística,0.201667,0.211752,0.201667,0.180421
6,AdaBoost (Boosting),0.181667,0.143371,0.181667,0.147154
